# NB25 — PAH Split Training: PAH Verisini Eğitime Dahil Etme Deneyleri

PAH panelinin %80/20 F1 skorunu 0.582'den (NB21 P4 COMBINED+BalBag) yükseltmek için
PAH verisinin bir kısmını eğitime dahil eden 4 deney.

## 4 Deney:

| # | Deney | Eğitim Havuzu | Test | Mantık |
|---|-------|---------------|------|--------|
| E1 | PAH-only | PAH-train (%50, ~186) | PAH-test (%50, ~186) | Minimal: sadece PAH ile ne yapabiliriz? |
| E2 | MASTER+PAH-half | MASTER(2931) + PAH-train(~186) | PAH-test | MASTER ağırlığı dominant kalmasın diye dengesiz: PAH eklemek fayda verir mi? |
| E3 | Balanced | ~186 MASTER + PAH-train(~186) | PAH-test | MASTER-PAH dengesi: her paneli eşit ağırlıkla eğit |
| E4 | NN/DNN Finetune | MASTER pretrain → PAH-train finetune | PAH-test | Deep learning transfer: pretrain+finetune stratejisi |

## Referans:
- NB21 P4 (COMBINED+BalBag) Boot F1=0.582 MCC=0.529 (%80/20)
- NB16 stack_lr PAH Boot F1=0.515

## Kritik Notlar:
1. **PAH split ONCE** — tüm deneylerde aynı train/test indices kullanılır
2. **M3 Preprocessing** — is_missing flags (>50% NaN) + median imputation
3. **Preprocessor fit only on train** — leakage kontrolü
4. **Cross-panel dup drop** — PAH'ın birebir-aynı MASTER satırları tespit ve remove
5. **Column cleanup** — constant cols + duplicate col pairs (MASTER'da tespit)
6. **Evaluation**: select_threshold_8020_robust() + bootstrap_8020()


In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedBaggingClassifier

# PyTorch (NN/DNN)
try:
    import torch
    import torch_directml
    DEVICE = torch.device("dml") if torch_directml.is_available() else torch.device("cpu")
except:
    import torch
    DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)

# Sabitler
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL
PI_TEST = 0.20           # final test: %20 pathogenic
FINAL_BENIGN_FRAC = 0.80 # final test: %80 benign
N_BOOT = 50              # bootstrap tekrar sayisi
BOOT_SEED = SEED
HIGH_MISS_THR = 0.50
NN_MAX_EPOCHS = 200
NN_PATIENCE = 15

# Sonuc dizini
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v9_pah_split_training")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f"NB25 -- PAH Split Training")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"DEVICE={DEVICE}")
print(f"Results -> {RESULTS_DIR}")

NB25 -- PAH Split Training
SEED=42, PI_TEST=0.2, N_BOOT=50
DEVICE=cpu
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v9_pah_split_training


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
# --- Veri yukleme ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"Ilk boyutlar:")
print(f"  MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"  KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"  CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"  PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- COMBINED: MASTER + KANSER + CFTR (PAH HARIC) ---
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+KANSER+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# --- Cross-panel exact-dup drop ---
feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tum feature+label birebir ayni olan satirlarin panel ID'lerini dondur."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_rows = panel_df.loc[panel_df[ID_COL] == vid, check_cols]
        if len(p_rows) == 0:
            continue
        p_row = p_rows.iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"PAH: {len(dup_ids)} birebir-ayni satir drop edildi -> {df_pah.shape}")
else:
    print(f"PAH: birebir-ayni satir yok")

# --- Sutun temizligi (MASTER uzerinde tespit) ---
# Sabit sutunlar
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

# Ozdes cift sutunlar
def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"\nSutun temizligi:")
print(f"  Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)}")
print(f"  Toplam drop: {len(drop_cols)}, Kalan: {len(feat_cols) - len(drop_cols)}")

# Drop
keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, PAH={df_pah.shape}")

Ilk boyutlar:
  MASTER: (2931, 353) (pos=2149, neg=782)
  KANSER: (388, 353) (pos=268, neg=120)
  CFTR:   (111, 353)   (pos=90, neg=21)
  PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+KANSER+CFTR): (3430, 353) (pos=2507, neg=923)
PAH: 3 birebir-ayni satir drop edildi -> (369, 353)

Sutun temizligi:
  Constant: 0, Duplicate pairs: 58
  Toplam drop: 58, Kalan: 293

Final shapes: MASTER=(2931, 295), COMBINED=(3430, 295), PAH=(369, 295)


In [3]:
# Cell 3: PAH Split + Preprocessing
# === PAH SPLIT ONCE ===
pah_train_idx, pah_test_idx = train_test_split(
    np.arange(len(df_pah)), test_size=0.5, random_state=SEED,
    stratify=df_pah[TARGET].values
)
pah_train_idx = np.sort(pah_train_idx)
pah_test_idx = np.sort(pah_test_idx)

df_pah_train = df_pah.iloc[pah_train_idx].reset_index(drop=True)
df_pah_test = df_pah.iloc[pah_test_idx].reset_index(drop=True)

y_pah_train = df_pah_train[TARGET].values
y_pah_test = df_pah_test[TARGET].values

print(f"PAH split:")
print(f"  Train: {df_pah_train.shape} (pos={y_pah_train.sum()}, neg={(y_pah_train==0).sum()})")
print(f"  Test:  {df_pah_test.shape} (pos={y_pah_test.sum()}, neg={(y_pah_test==0).sum()})")

# --- Preprocessing fonksiyonlari ---
def fit_preprocessor(train_df, keep_cols, target, high_miss_thr=HIGH_MISS_THR):
    """Train uzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > high_miss_thr].index.tolist()
    
    # Median (train uzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sutunlar icin)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

print("\nPreprocessor fit and transform...")
print("Tum deneyler hazir.")

PAH split:
  Train: (184, 295) (pos=153, neg=31)
  Test:  (185, 295) (pos=154, neg=31)

Preprocessor fit and transform...
Tum deneyler hazir.


In [4]:
# Cell 4: Evaluation Altyapisi
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    """Saerens prior-shift duzeltmesi."""
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    """%80 benign / %20 patho resampling."""
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    """Bootstrap %80/20 F1 with CI."""
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {
        "mean": float(f1s.mean()), "std": float(f1s.std()),
        "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))
    }

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    """Robust threshold selection on resampled 80/20 train data."""
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

def eval_holdout(y_train, p_train, y_test, p_test, pi_train=None, prior_shift=False):
    """Holdout evaluation with all metrics."""
    if pi_train is None:
        pi_train = y_train.mean()
    
    prob = adjust_prior_shift(p_test, pi_train=pi_train) if prior_shift else p_test
    thr = select_threshold_8020_robust(y_train, p_train if not prior_shift else adjust_prior_shift(p_train, pi_train))
    
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_test, y_pred) if len(np.unique(y_test)) > 1 else 0.0
    f1 = _f1_pos(y_test, y_pred)
    auc = roc_auc_score(y_test, prob) if len(np.unique(y_test)) > 1 else 0.0
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_test, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    
    # Train metrikleri
    p_train_adj = adjust_prior_shift(p_train, pi_train) if prior_shift else p_train
    yp_train = (p_train_adj >= thr).astype(int)
    train_f1 = _f1_pos(y_train, yp_train)
    train_mcc = matthews_corrcoef(y_train, yp_train) if len(np.unique(y_train)) > 1 else 0.0
    
    return {
        "mcc": float(mcc), "f1": float(f1), "auc": float(auc),
        "precision": float(prec), "recall": float(rec),
        "thr": float(thr), "boot8020": boot,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "train_f1": float(train_f1), "train_mcc": float(train_mcc),
        "pi_train": float(pi_train)
    }

print("Evaluation altyapisi hazir.")

Evaluation altyapisi hazir.


In [5]:
# Cell 5: Model Yardimlari
LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def _le_encode(X_df):
    """Label-encode kategorik sutunlar."""
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

# SmallMLP & FocalLoss for NN/DNN
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        targets = targets.float()
        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none",
            pos_weight=torch.tensor([self.pos_weight], device=logits.device)
        )
        p_t = torch.sigmoid(logits)
        p_t = torch.where(targets == 1, p_t, 1 - p_t)
        focal_w = (1 - p_t) ** self.gamma
        return (self.alpha * focal_w * bce).mean()

class SmallMLP(torch.nn.Module):
    def __init__(self, input_dim, hidden, n_layers, dropout):
        super().__init__()
        layers = []
        d = input_dim
        for _ in range(n_layers):
            layers += [
                torch.nn.Linear(d, hidden),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout)
            ]
            d = hidden
        layers += [torch.nn.Linear(d, 1)]
        self.net = torch.nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

def _train_nn_es(model, X_tr, y_tr, X_val, y_val, combo, pos_weight, max_epochs=NN_MAX_EPOCHS, patience=NN_PATIENCE):
    """Train NN with early stopping on validation F1."""
    X_tr_t = torch.from_numpy(X_tr).float().to(DEVICE)
    y_tr_t = torch.from_numpy(y_tr).float().to(DEVICE)
    X_val_t = torch.from_numpy(X_val).float().to(DEVICE)
    y_val_t = torch.from_numpy(y_val).float().to(DEVICE)
    
    crit = FocalLoss(alpha=0.25, gamma=combo["focal_gamma"], pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=combo["lr"], weight_decay=combo["weight_decay"])
    
    n = len(X_tr_t)
    bs = min(64, max(2, n - 1))
    best_f1, best_state, pat = -1.0, None, 0
    
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss = crit(model(X_tr_t[idx]), y_tr_t[idx])
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(X_val_t)).cpu().numpy()
        vf1 = _f1_pos(y_val_t.cpu().numpy(), (vp >= 0.5).astype(int))
        
        if vf1 > best_f1:
            best_f1, best_state, pat = vf1, deepcopy(model.state_dict()), 0
        else:
            pat += 1
            if pat >= patience:
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

print("Model yardimlari hazir.")

Model yardimlari hazir.


In [6]:
# Cell 6: SMOKE TEST
print("\n" + "="*70)
print("SMOKE TEST")
print("="*70)

# 1. Verify PAH split
assert len(pah_train_idx) + len(pah_test_idx) == len(df_pah), "PAH split error"
assert len(set(pah_train_idx) & set(pah_test_idx)) == 0, "PAH train/test overlap"
print(f"\n1. PAH split OK:")
print(f"   Train: {len(pah_train_idx)} (pos={y_pah_train.sum()}, neg={(y_pah_train==0).sum()})")
print(f"   Test:  {len(pah_test_idx)} (pos={y_pah_test.sum()}, neg={(y_pah_test==0).sum()})")

# 2. Quick preprocessing
prep_e1 = fit_preprocessor(df_pah_train, keep_cols, TARGET)
X_e1_train = transform_X(df_pah_train, keep_cols, prep_e1)
X_e1_test = transform_X(df_pah_test, keep_cols, prep_e1)
print(f"\n2. Preprocessing OK:")
print(f"   X_e1_train: {X_e1_train.shape}")
print(f"   X_e1_test: {X_e1_test.shape}")

# 3. Label encode
X_e1_train_le, _ = _le_encode(X_e1_train)
X_e1_test_le, _ = _le_encode(X_e1_test)
print(f"\n3. Label encode OK: {X_e1_train_le.shape} -> numeric")

# 4. Quick LightGBM (n_estimators=5)
m_quick = _lgbm_classifier(n_estimators=5, verbose=-1)
m_quick.fit(X_e1_train_le, y_pah_train)
p_quick = m_quick.predict_proba(X_e1_test_le)[:, 1]
auc_quick = roc_auc_score(y_pah_test, p_quick) if len(np.unique(y_pah_test)) > 1 else 0.0
print(f"\n4. Quick LightGBM OK:")
print(f"   E1 (PAH-only) quick AUC: {auc_quick:.4f}")

print("\n" + "="*70)
print("SMOKE TEST PASSED")
print("="*70)


SMOKE TEST

1. PAH split OK:
   Train: 184 (pos=153, neg=31)
   Test:  185 (pos=154, neg=31)

2. Preprocessing OK:
   X_e1_train: (184, 471)
   X_e1_test: (185, 471)

3. Label encode OK: (184, 471) -> numeric

4. Quick LightGBM OK:
   E1 (PAH-only) quick AUC: 0.7608

SMOKE TEST PASSED


In [7]:
# Cell 7: Experiments E1-E3 (LightGBM)
print("\n" + "="*70)
print("E1-E3: LightGBM Experiments")
print("="*70)

all_results = {}

# ================================================================
# E1: PAH-only
# ================================================================
print("\nE1: PAH-only LightGBM...")
prep_e1 = fit_preprocessor(df_pah_train, keep_cols, TARGET)
X_e1_train = transform_X(df_pah_train, keep_cols, prep_e1)
X_e1_test = transform_X(df_pah_test, keep_cols, prep_e1)
X_e1_train_le, _ = _le_encode(X_e1_train)
X_e1_test_le, _ = _le_encode(X_e1_test)

m_e1 = _lgbm_classifier()
m_e1.fit(X_e1_train_le, y_pah_train)
p_e1_train = m_e1.predict_proba(X_e1_train_le)[:, 1]
p_e1_test = m_e1.predict_proba(X_e1_test_le)[:, 1]

pi_e1_train = float(y_pah_train.mean())
res_e1 = eval_holdout(y_pah_train, p_e1_train, y_pah_test, p_e1_test, pi_train=pi_e1_train, prior_shift=True)
all_results["E1_PAH-only"] = {**res_e1, "n_train": len(df_pah_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e1['mcc']:.4f}, Boot-mean={res_e1['boot8020']['mean']:.4f}")

# E1 with BalancedBagging
print("E1b: PAH-only BalancedBagging...")
bb_e1 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_e1.fit(X_e1_train_le, y_pah_train)
p_e1b_train = bb_e1.predict_proba(X_e1_train_le)[:, 1]
p_e1b_test = bb_e1.predict_proba(X_e1_test_le)[:, 1]
res_e1b = eval_holdout(y_pah_train, p_e1b_train, y_pah_test, p_e1b_test, pi_train=pi_e1_train, prior_shift=True)
all_results["E1b_PAH-only_BalBag"] = {**res_e1b, "n_train": len(df_pah_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e1b['mcc']:.4f}, Boot-mean={res_e1b['boot8020']['mean']:.4f}")

# ================================================================
# E2: MASTER + PAH-half
# ================================================================
print("\nE2: MASTER+PAH-train LightGBM...")
df_e2_train = pd.concat([df_master, df_pah_train], ignore_index=True)
y_e2_train = df_e2_train[TARGET].values

prep_e2 = fit_preprocessor(df_e2_train, keep_cols, TARGET)
X_e2_train = transform_X(df_e2_train, keep_cols, prep_e2)
X_e2_test = transform_X(df_pah_test, keep_cols, prep_e2)
X_e2_train_le, _ = _le_encode(X_e2_train)
X_e2_test_le, _ = _le_encode(X_e2_test)

m_e2 = _lgbm_classifier()
m_e2.fit(X_e2_train_le, y_e2_train)
p_e2_train = m_e2.predict_proba(X_e2_train_le)[:, 1]
p_e2_test = m_e2.predict_proba(X_e2_test_le)[:, 1]

pi_e2_train = float(y_e2_train.mean())
res_e2 = eval_holdout(y_e2_train, p_e2_train, y_pah_test, p_e2_test, pi_train=pi_e2_train, prior_shift=True)
all_results["E2_MASTER+PAH"] = {**res_e2, "n_train": len(df_e2_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e2['mcc']:.4f}, Boot-mean={res_e2['boot8020']['mean']:.4f}")

# E2 with BalancedBagging
print("E2b: MASTER+PAH-train BalancedBagging...")
bb_e2 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_e2.fit(X_e2_train_le, y_e2_train)
p_e2b_train = bb_e2.predict_proba(X_e2_train_le)[:, 1]
p_e2b_test = bb_e2.predict_proba(X_e2_test_le)[:, 1]
res_e2b = eval_holdout(y_e2_train, p_e2b_train, y_pah_test, p_e2b_test, pi_train=pi_e2_train, prior_shift=True)
all_results["E2b_MASTER+PAH_BalBag"] = {**res_e2b, "n_train": len(df_e2_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e2b['mcc']:.4f}, Boot-mean={res_e2b['boot8020']['mean']:.4f}")

# ================================================================
# E3: Balanced (equal MASTER + PAH-train)
# ================================================================
print("\nE3: Balanced (equal MASTER+PAH-train) LightGBM...")
# Sample ~186 from MASTER (stratified, same pos/neg ratio)
pos_idx_master = np.where(df_master[TARGET] == 1)[0]
neg_idx_master = np.where(df_master[TARGET] == 0)[0]
n_sample = len(df_pah_train)
pos_ratio = y_pah_train.mean()
n_pos_sample = int(n_sample * pos_ratio)
n_neg_sample = n_sample - n_pos_sample

rng = np.random.RandomState(SEED)
pos_sample_idx = rng.choice(pos_idx_master, size=min(n_pos_sample, len(pos_idx_master)), replace=False)
neg_sample_idx = rng.choice(neg_idx_master, size=min(n_neg_sample, len(neg_idx_master)), replace=False)
master_sample_idx = np.concatenate([pos_sample_idx, neg_sample_idx])
df_master_sample = df_master.iloc[master_sample_idx].reset_index(drop=True)

df_e3_train = pd.concat([df_master_sample, df_pah_train], ignore_index=True)
y_e3_train = df_e3_train[TARGET].values

prep_e3 = fit_preprocessor(df_e3_train, keep_cols, TARGET)
X_e3_train = transform_X(df_e3_train, keep_cols, prep_e3)
X_e3_test = transform_X(df_pah_test, keep_cols, prep_e3)
X_e3_train_le, _ = _le_encode(X_e3_train)
X_e3_test_le, _ = _le_encode(X_e3_test)

m_e3 = _lgbm_classifier()
m_e3.fit(X_e3_train_le, y_e3_train)
p_e3_train = m_e3.predict_proba(X_e3_train_le)[:, 1]
p_e3_test = m_e3.predict_proba(X_e3_test_le)[:, 1]

pi_e3_train = float(y_e3_train.mean())
res_e3 = eval_holdout(y_e3_train, p_e3_train, y_pah_test, p_e3_test, pi_train=pi_e3_train, prior_shift=True)
all_results["E3_Balanced"] = {**res_e3, "n_train": len(df_e3_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e3['mcc']:.4f}, Boot-mean={res_e3['boot8020']['mean']:.4f}")

# E3 with BalancedBagging
print("E3b: Balanced BalancedBagging...")
bb_e3 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_e3.fit(X_e3_train_le, y_e3_train)
p_e3b_train = bb_e3.predict_proba(X_e3_train_le)[:, 1]
p_e3b_test = bb_e3.predict_proba(X_e3_test_le)[:, 1]
res_e3b = eval_holdout(y_e3_train, p_e3b_train, y_pah_test, p_e3b_test, pi_train=pi_e3_train, prior_shift=True)
all_results["E3b_Balanced_BalBag"] = {**res_e3b, "n_train": len(df_e3_train), "n_test": len(df_pah_test)}
print(f"  MCC={res_e3b['mcc']:.4f}, Boot-mean={res_e3b['boot8020']['mean']:.4f}")

print("\nE1-E3 tamamlandi!")


E1-E3: LightGBM Experiments

E1: PAH-only LightGBM...
  MCC=0.4227, Boot-mean=0.4904
E1b: PAH-only BalancedBagging...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.3902, Boot-mean=0.5648

E2: MASTER+PAH-train LightGBM...
  MCC=0.5694, Boot-mean=0.4979
E2b: MASTER+PAH-train BalancedBagging...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.5114, Boot-mean=0.5075

E3: Balanced (equal MASTER+PAH-train) LightGBM...
  MCC=0.3411, Boot-mean=0.4139
E3b: Balanced BalancedBagging...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.2648, Boot-mean=0.4426

E1-E3 tamamlandi!


In [ ]:
# Cell 8: E4 — NN/DNN Finetune
print("\n" + "="*70)
print("E4: NN/DNN Finetune")
print("="*70)

# KRITIK: Hem MASTER hem PAH ayni preprocessor ile transform edilmeli.
# NB15 yaklasimi: fit_basis = concat([pretrain_df, panel_train_df])
df_e4_basis = pd.concat([df_master, df_pah_train], ignore_index=True)
prep_e4 = fit_preprocessor(df_e4_basis, keep_cols, TARGET)

X_master_e4 = transform_X(df_master, keep_cols, prep_e4)
X_pah_train_e4 = transform_X(df_pah_train, keep_cols, prep_e4)
X_pah_test_e4 = transform_X(df_pah_test, keep_cols, prep_e4)
y_master_e4 = df_master[TARGET].values

print(f"E4 feature space: MASTER={X_master_e4.shape}, PAH-train={X_pah_train_e4.shape}, PAH-test={X_pah_test_e4.shape}")

# Sayisal matrise cevir (NN icin LE + float32)
X_master_e4_le, le_maps_e4 = _le_encode(X_master_e4)
X_pah_train_e4_le, _ = _le_encode(X_pah_train_e4)
X_pah_test_e4_le, _ = _le_encode(X_pah_test_e4)

X_master_np = X_master_e4_le.values.astype(np.float32)
X_pah_train_np = X_pah_train_e4_le.values.astype(np.float32)
X_pah_test_np = X_pah_test_e4_le.values.astype(np.float32)

# NN/DNN config gridi
NN_CONFIGS = [
    {"n_layers": 2, "hidden": 64, "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-3, "focal_gamma": 2.0, "name": "NN-L2-H64"},
    {"n_layers": 2, "hidden": 128, "dropout": 0.5, "lr": 1e-3, "weight_decay": 1e-3, "focal_gamma": 2.0, "name": "NN-L2-H128"},
    {"n_layers": 3, "hidden": 128, "dropout": 0.4, "lr": 1e-3, "weight_decay": 1e-3, "focal_gamma": 2.0, "name": "DNN-L3-H128"},
]

# ================================================================
# Adim 1: Pretrain on MASTER (5-fold'un ilk fold'u validation)
# ================================================================
print(f"\nPretrain on MASTER (n={len(X_master_np)})...")

skf_pre = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pre_tr_idx, pre_val_idx = next(iter(skf_pre.split(X_master_np, y_master_e4)))

X_pre_tr, y_pre_tr = X_master_np[pre_tr_idx], y_master_e4[pre_tr_idx]
X_pre_val, y_pre_val = X_master_np[pre_val_idx], y_master_e4[pre_val_idx]

best_pretrain = None
best_pretrain_f1 = -1.0
best_config = None

for config in NN_CONFIGS:
    print(f"\n  Pretraining {config['name']}...")
    model = SmallMLP(
        input_dim=X_pre_tr.shape[1],
        hidden=config["hidden"],
        n_layers=config["n_layers"],
        dropout=config["dropout"]
    ).to(DEVICE)
    
    pw = float((y_pre_tr == 0).sum()) / max(1, float((y_pre_tr == 1).sum()))
    model = _train_nn_es(model, X_pre_tr, y_pre_tr, X_pre_val, y_pre_val, config, pw)
    
    model.eval()
    with torch.no_grad():
        vp = torch.sigmoid(model(torch.from_numpy(X_pre_val).float().to(DEVICE))).cpu().numpy()
    vf1 = _f1_pos(y_pre_val, (vp >= 0.5).astype(int))
    print(f"    Pretrain val F1: {vf1:.4f}")
    
    if vf1 > best_pretrain_f1:
        best_pretrain_f1, best_pretrain, best_config = vf1, deepcopy(model), config

print(f"\nBest pretrain config: {best_config['name']} (val F1={best_pretrain_f1:.4f})")

# ================================================================
# Adim 2: Finetune on PAH-train (dusuk lr)
# ================================================================
print(f"\nFinetune on PAH-train (n={len(X_pah_train_np)})...")

skf_ft = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ft_tr_idx, ft_val_idx = next(iter(skf_ft.split(X_pah_train_np, y_pah_train)))

X_ft_tr, y_ft_tr = X_pah_train_np[ft_tr_idx], y_pah_train[ft_tr_idx]
X_ft_val, y_ft_val = X_pah_train_np[ft_val_idx], y_pah_train[ft_val_idx]

ft_config = {**best_config, "lr": best_config["lr"] * 0.1}
model_ft = deepcopy(best_pretrain)
pw_ft = float((y_ft_tr == 0).sum()) / max(1, float((y_ft_tr == 1).sum()))
model_ft = _train_nn_es(model_ft, X_ft_tr, y_ft_tr, X_ft_val, y_ft_val, ft_config, pw_ft)

# Predict
model_ft.eval()
with torch.no_grad():
    p_ft_train = torch.sigmoid(model_ft(torch.from_numpy(X_pah_train_np).float().to(DEVICE))).cpu().numpy()
    p_ft_test = torch.sigmoid(model_ft(torch.from_numpy(X_pah_test_np).float().to(DEVICE))).cpu().numpy()

pi_e4 = float(y_pah_train.mean())
res_e4 = eval_holdout(y_pah_train, p_ft_train, y_pah_test, p_ft_test, pi_train=pi_e4, prior_shift=True)
all_results["E4_NN_Finetune_MASTER"] = {**res_e4, "n_train": len(df_pah_train), "n_test": len(df_pah_test)}
print(f"E4a (MASTER pretrain) Result: MCC={res_e4['mcc']:.4f}, Boot-mean={res_e4['boot8020']['mean']:.4f}")

# ================================================================
# Adim 3: COMBINED pretrain -> PAH finetune (alternatif)
# ================================================================
print(f"\nE4b: COMBINED pretrain -> PAH finetune...")
df_e4b_basis = pd.concat([df_combined, df_pah_train], ignore_index=True)
prep_e4b = fit_preprocessor(df_e4b_basis, keep_cols, TARGET)

X_comb_e4b = transform_X(df_combined, keep_cols, prep_e4b)
X_pah_tr_e4b = transform_X(df_pah_train, keep_cols, prep_e4b)
X_pah_te_e4b = transform_X(df_pah_test, keep_cols, prep_e4b)

X_comb_le, _ = _le_encode(X_comb_e4b)
X_ptr_le, _ = _le_encode(X_pah_tr_e4b)
X_pte_le, _ = _le_encode(X_pah_te_e4b)

X_comb_np = X_comb_le.values.astype(np.float32)
X_ptr_np = X_ptr_le.values.astype(np.float32)
X_pte_np = X_pte_le.values.astype(np.float32)
y_comb = df_combined[TARGET].values

# Pretrain on COMBINED
pre_tr2, pre_val2 = next(iter(skf_pre.split(X_comb_np, y_comb)))
model_b = SmallMLP(
    input_dim=X_comb_np.shape[1],
    hidden=best_config["hidden"],
    n_layers=best_config["n_layers"],
    dropout=best_config["dropout"]
).to(DEVICE)
pw_b = float((y_comb[pre_tr2] == 0).sum()) / max(1, float((y_comb[pre_tr2] == 1).sum()))
model_b = _train_nn_es(model_b, X_comb_np[pre_tr2], y_comb[pre_tr2],
                        X_comb_np[pre_val2], y_comb[pre_val2], best_config, pw_b)

# Finetune on PAH-train
ft_tr2, ft_val2 = next(iter(skf_ft.split(X_ptr_np, y_pah_train)))
ft_config_b = {**best_config, "lr": best_config["lr"] * 0.1}
pw_fb = float((y_pah_train[ft_tr2] == 0).sum()) / max(1, float((y_pah_train[ft_tr2] == 1).sum()))
model_b = _train_nn_es(model_b, X_ptr_np[ft_tr2], y_pah_train[ft_tr2],
                        X_ptr_np[ft_val2], y_pah_train[ft_val2], ft_config_b, pw_fb)

model_b.eval()
with torch.no_grad():
    p_b_train = torch.sigmoid(model_b(torch.from_numpy(X_ptr_np).float().to(DEVICE))).cpu().numpy()
    p_b_test = torch.sigmoid(model_b(torch.from_numpy(X_pte_np).float().to(DEVICE))).cpu().numpy()

res_e4b = eval_holdout(y_pah_train, p_b_train, y_pah_test, p_b_test, pi_train=pi_e4, prior_shift=True)
all_results["E4b_NN_Finetune_COMBINED"] = {**res_e4b, "n_train": len(df_pah_train), "n_test": len(df_pah_test)}
print(f"E4b (COMBINED pretrain) Result: MCC={res_e4b['mcc']:.4f}, Boot-mean={res_e4b['boot8020']['mean']:.4f}")

print("\nE4 tamamlandi!")



E4: NN/DNN Finetune
E4 feature space: MASTER=(2931, 434), PAH-train=(184, 434), PAH-test=(185, 434)

Pretrain on MASTER (n=2931)...

  Pretraining NN-L2-H64...
    Pretrain val F1: 0.8456

  Pretraining NN-L2-H128...
    Pretrain val F1: 0.8456

  Pretraining DNN-L3-H128...
    Pretrain val F1: 0.8456

Best pretrain config: NN-L2-H64 (val F1=0.8456)

Finetune on PAH-train (n=184)...
E4a (MASTER pretrain) Result: MCC=0.0000, Boot-mean=0.3404

E4b: COMBINED pretrain -> PAH finetune...


In [ ]:
# Cell 9: Results Summary Table
print("\n" + "="*70)
print("NB25 SONUC DERLEMESI")
print("="*70)

rows = []
for name, res in all_results.items():
    boot = res["boot8020"]
    rows.append({
        "Deney": name,
        "n_train": res["n_train"],
        "MCC": round(res["mcc"], 4),
        "F1": round(res["f1"], 4),
        "AUC": round(res["auc"], 4),
        "Precision": round(res["precision"], 4),
        "Recall": round(res["recall"], 4),
        "Boot-mean": round(boot["mean"], 4),
        "Boot-std": round(boot["std"], 4),
        "Boot-lo": round(boot["lo"], 4),
        "Boot-hi": round(boot["hi"], 4),
        "Threshold": round(res["thr"], 3),
        "Train-F1": round(res["train_f1"], 4),
        "Train-MCC": round(res["train_mcc"], 4),
        "TN": res["tn"],
        "FP": res["fp"],
        "FN": res["fn"],
        "TP": res["tp"]
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("Boot-mean", ascending=False).reset_index(drop=True)

print("\n=== Tum Deneylerin Sonuclari (Boot-mean siralama) ===")
display_cols = ["Deney", "n_train", "MCC", "Boot-mean", "Boot-std", 
                "Precision", "Recall", "Threshold", "TN", "FP", "FN", "TP", "Train-F1"]
print("\n" + results_df[display_cols].to_string(index=False))

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "pah_split_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nSonuclar kaydedildi: {csv_path}")

# Referans karsilastirmasi
print(f"\n=== Referans Karsilastirmasi ===")
print(f"NB21 P4 (COMBINED+BalBag): Boot F1=0.582 MCC=0.529")
print(f"NB16 stack_lr: Boot F1=0.515")
best_boot = results_df.iloc[0]
delta_nb21 = best_boot["Boot-mean"] - 0.582
delta_nb16 = best_boot["Boot-mean"] - 0.515
print(f"\nEn iyi NB25: {best_boot['Deney']} Boot F1={best_boot['Boot-mean']:.4f}")
print(f"  Delta vs NB21: {delta_nb21:+.4f}")
print(f"  Delta vs NB16: {delta_nb16:+.4f}")

In [ ]:
# Cell 10: Visualizations
print("\n" + "="*70)
print("GORSELLESTIRMELER")
print("="*70)

# Fig 1: Boot-mean comparison
fig, ax = plt.subplots(figsize=(12, 6))
names = results_df["Deney"].tolist()
boots = results_df["Boot-mean"].tolist()
stds = results_df["Boot-std"].tolist()
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))

x = np.arange(len(names))
bars = ax.bar(x, boots, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor="black", linewidth=1.5)
ax.set_ylabel("Bootstrap %80/20 F1 (pathogenic)", fontsize=11)
ax.set_title("NB25 — PAH Split Training: Bootstrap %80/20 F1 Comparison", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.axhline(y=0.582, color="red", linestyle="--", alpha=0.6, linewidth=2, label="NB21 P4 (0.582)")
ax.axhline(y=0.515, color="blue", linestyle="--", alpha=0.6, linewidth=2, label="NB16 (0.515)")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
for b, val in zip(bars, boots):
    ax.annotate(f"{val:.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_boot_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig1_boot_comparison.png kaydedildi")

# Fig 2: MCC comparison
fig, ax = plt.subplots(figsize=(12, 6))
mccs = results_df["MCC"].tolist()
bars = ax.bar(x, mccs, color=colors, alpha=0.85, edgecolor="black", linewidth=1.5)
ax.set_ylabel("Matthews Correlation Coefficient (MCC)", fontsize=11)
ax.set_title("NB25 — PAH: MCC Comparison", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.axhline(y=0.529, color="red", linestyle="--", alpha=0.6, linewidth=2, label="NB21 P4 (0.529)")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
for b, val in zip(bars, mccs):
    ax.annotate(f"{val:.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_mcc_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig2_mcc_comparison.png kaydedildi")

# Fig 3: Confusion matrices (top 3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i in range(min(3, len(results_df))):
    row = results_df.iloc[i]
    cm = np.array([
        [row["TN"], row["FP"]],
        [row["FN"], row["TP"]]
    ])
    ax = axes[i]
    im = ax.imshow(cm, cmap="Blues", aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Pathogenic"])
    ax.set_yticklabels(["Benign", "Pathogenic"])
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True", fontsize=10)
    ax.set_title(f"{row['Deney'][:20]}\nMCC={row['MCC']:.3f}, F1={row['F1']:.3f}", fontsize=9, fontweight="bold")
    for ii in range(2):
        for jj in range(2):
            ax.text(jj, ii, str(int(cm[ii, jj])), ha="center", va="center", fontsize=16,
                   color="white" if cm[ii, jj] > cm.max()/2 else "black", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig3_confusion_matrices.png kaydedildi")

# Fig 4: Train vs test gap (overfit check)
fig, ax = plt.subplots(figsize=(12, 6))
train_f1s = results_df["Train-F1"].tolist()
test_f1s = results_df["F1"].tolist()
gaps = [t - e for t, e in zip(train_f1s, test_f1s)]

x_pos = np.arange(len(names))
width = 0.35
ax.bar(x_pos - width/2, train_f1s, width, label="Train F1", color="steelblue", alpha=0.8, edgecolor="black")
ax.bar(x_pos + width/2, test_f1s, width, label="Test F1", color="darkorange", alpha=0.8, edgecolor="black")
ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("NB25 — Overfit Check: Train vs Test F1", fontsize=13, fontweight="bold")
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_overfit_check.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig4_overfit_check.png kaydedildi")

print("\nTum gorsellestirmeler tamamlandi!")

In [ ]:
# Cell 11: PDF Report
print("\n" + "="*70)
print("PDF RAPOR OLUSTURMA")
print("="*70)

from fpdf import FPDF

class PAHSplitReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "PAH Split Training Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(200, 50, 50)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)
    
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")
    
    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
    
    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.ln(2)
    
    def add_table(self, headers, data, col_widths=None):
        if col_widths is None:
            col_widths = [190 / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 7)
        self.set_fill_color(0, 102, 153)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, h, 1, 0, "C", True)
        self.ln()
        self.set_x(self.l_margin)
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 6.5)
        for row in data:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C")
            self.ln()
            self.set_x(self.l_margin)
    
    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210-w)/2, w=w)
            self.ln(3)

pdf = PAHSplitReport()
pdf.alias_nb_pages()
pdf.add_page()

# Baslik
pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "PAH Split Training: 4 Deney", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB25 | {datetime.now().strftime('%Y-%m-%d %H:%M')}", 0, 1, "C")
pdf.ln(5)

# Yonetici ozeti
pdf.section("Yonetici Ozeti")
best = results_df.iloc[0]
pdf.body(
    f"PAH panelinin performansini iyilestirmek icin PAH verisinin bir kismi "
    f"egitim setine dahil edilip 4 deney yapildi. En iyi strateji: {best['Deney']} "
    f"(Boot-mean=%80/20 F1={best['Boot-mean']:.4f}, MCC={best['MCC']:.4f}). "
    f"NB21 P4 referansi: Boot F1=0.582, MCC=0.529."
)

# Sonuc tablosu
pdf.section("1. Deney Sonuclari")
headers = ["Deney", "n_train", "MCC", "Boot-mean", "Boot-std", "F1", "Prec", "Rec", "TN", "FP", "FN", "TP"]
cw = [25, 15, 14, 16, 14, 12, 12, 12, 10, 10, 10, 10]
data = []
for _, row in results_df.iterrows():
    data.append([
        row["Deney"][:20], int(row["n_train"]), 
        f"{row['MCC']:.3f}", f"{row['Boot-mean']:.3f}", f"{row['Boot-std']:.3f}",
        f"{row['F1']:.3f}", f"{row['Precision']:.3f}", f"{row['Recall']:.3f}",
        int(row["TN"]), int(row["FP"]), int(row["FN"]), int(row["TP"])
    ])
pdf.add_table(headers, data, cw)
pdf.ln(3)

# Figurler
pdf.section("2. Bootstrap %80/20 F1 Karsilastirmasi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_boot_comparison.png"))

pdf.add_page()
pdf.section("3. MCC Karsilastirmasi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_mcc_comparison.png"))

pdf.section("4. Confusion Matrix (En Iyi 3 Deney)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_confusion_matrices.png"))

pdf.add_page()
pdf.section("5. Overfit Kontrolu (Train vs Test F1)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_overfit_check.png"))

# Tartisma
pdf.section("6. Tartisma")
pdf.body(
    "4 deney yapilmistir: (E1) PAH-only, (E2) MASTER+PAH-half, (E3) balanced (equal MASTER+PAH), "
    "ve (E4) NN/DNN finetune. Tum deneylerde M3 preprocessing (is_missing flags + median imputation) "
    "ve %80/20 benign-aware threshold oturumu kullanilmistir."
)
pdf.body(
    f"En iyi sonuc: {best['Deney']} ile Boot-mean %80/20 F1={best['Boot-mean']:.4f} "
    f"(CI=[{best['Boot-lo']:.4f}, {best['Boot-hi']:.4f}]). "
    f"Delta vs NB21 P4: {best['Boot-mean']-0.582:+.4f}. "
    f"Overfit kontrolu: train-test F1 gap={best['Train-F1']-best['F1']:.4f}."
)

# Kaydet
pdf_path = os.path.join(REPORTS_DIR_NB, "NB25_pah_split_training_report.pdf")
pdf.output(pdf_path)
print(f"PDF raporu olusturuldu: {pdf_path}")